# OpenSSF Scorecard

_Part of the **Decay Before Archival** project — see `README.md` for full project context, setup instructions, and links to the other notebooks (`deps_dev.ipynb`, `scorecard.ipynb`, `gh_archive.ipynb`, `pypi.ipynb`, `other_sources.ipynb`)._

In [ ]:
from google.cloud import bigquery
from decay_before_archival import get_client

client = get_client()

## Explore: OpenSSF Scorecard

Same pattern as deps.dev — list the tables in `openssf.scorecardcron`, then peek columns on whichever table looks most relevant, before writing real queries. (Some table names in this dataset include hyphens, which need backticks in raw SQL — keep an eye out.)

**Planned analysis:** per-check health scores (e.g. maintenance activity, code review practices, CI coverage) as direct health features, plus an overall score as a comparison baseline against our own risk score.

In [ ]:
tables_query = """
SELECT table_name, table_type
FROM `openssf.scorecardcron.INFORMATION_SCHEMA.TABLES`
ORDER BY table_name
"""
tables_dry_run = client.query(
    tables_query,
    job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False),
)
estimated_bytes = tables_dry_run.total_bytes_processed or 0
print(f"Metadata estimate: { estimated_bytes / 1e6:.2f} MB")
scorecard_tables = client.query(tables_query).to_dataframe()
scorecard_tables

In [ ]:
scorecard_table = "scorecard-v2"

columns_query = """
SELECT column_name, field_path, data_type
FROM `openssf.scorecardcron.INFORMATION_SCHEMA.COLUMN_FIELD_PATHS`
WHERE table_name = @table_name
ORDER BY field_path
"""
columns_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("table_name", "STRING", scorecard_table)
    ],
)
scorecard_columns = client.query(
    columns_query,
    job_config=columns_config,
).to_dataframe()
scorecard_columns

## Read the GH Archive CSV file & Retrieve Scorecard Information

In [ ]:
import pandas as pd

gh_clean = pd.read_csv(
    "../data/gh_archive_results_clean.csv",
    dtype={"repo_id": "string"},
)

gh_clean["repo_key"] = (
    gh_clean["repo_full_name"].astype("string").str.strip().str.lower()
)
repo_names = gh_clean["repo_key"].dropna().drop_duplicates().tolist()

# Preview a small, cutoff-bounded sample after confirming the schema above.
sample_query = """
WITH requested AS (
  SELECT repo_name
  FROM UNNEST(@repo_names) AS repo_name
),
ranked AS (
  SELECT
    requested.repo_name,
    sc.date AS scorecard_date,
    sc.score AS scorecard_score,
    ROW_NUMBER() OVER (
      PARTITION BY requested.repo_name
      ORDER BY sc.date DESC
    ) AS scorecard_rank
  FROM requested
  LEFT JOIN `openssf.scorecardcron.scorecard-v2` AS sc
    ON LOWER(TRIM(sc.repo.name)) =
       CONCAT('github.com/', requested.repo_name)
    AND sc.date < DATE '2025-01-01'
)
SELECT
  repo_name,
  scorecard_date,
  scorecard_score,
  scorecard_date IS NOT NULL AS has_scorecard_match
FROM ranked
WHERE scorecard_rank = 1
"""
sample_dry_run_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ArrayQueryParameter("repo_names", "STRING", repo_names)
    ],
    dry_run=True,
    use_query_cache=False,
)

sample_dry_run = client.query(sample_query, job_config=sample_dry_run_config)
estimated_bytes = sample_dry_run.total_bytes_processed or 0
print(f"Estimated scan: {estimated_bytes / 1e9:.2f} GB")

# Execute the sample only after checking the estimate.
sample_config = bigquery.QueryJobConfig(
    query_parameters=[
            bigquery.ArrayQueryParameter("repo_names", "STRING", repo_names)
    ],
    use_query_cache=False
)
scorecard_sample = client.query(
    sample_query,
    job_config=sample_config,
).to_dataframe()

print("Rows:", len(scorecard_sample))

scorecard_sample.to_csv(
    "../data/scorecard_gh_sample.csv",
    index=False,
)

**Notes:**

Estimated scan for the sample query was 7.63GB. Returned 62,581 rows but only about 287 rows have openssf scorecard information.

## Viewing Missing Values

In [ ]:
import pandas as pd

scorecard_raw = pd.read_csv(
    "../data/scorecard_gh_sample.csv",
    dtype={"repo_id": "string"},
    low_memory=False,
)

missing_audit = pd.DataFrame({
    "dtype": scorecard_raw.dtypes.astype(str),
    "missing_count": scorecard_raw.isna().sum(),
    "missing_pct": (scorecard_raw.isna().mean() * 100).round(2),
    "unique_values": scorecard_raw.nunique(dropna=True),
}).sort_values("missing_count", ascending=False)

display(missing_audit)

**Notes:**

_(add observations here)_